In [1]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np 

In [2]:
df = pd.read_csv('03_Churn_Modelling.csv')

In [3]:
df.head()

,apst,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,delhi,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,bangalore,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,delhi,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,delhi,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,bangalore,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
df.isnull().sum()

apst               0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [5]:
X = df.iloc[:,3:-1].values
y = df.iloc[:,-1].values

In [6]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X[:,2] = le.fit_transform(X[:,2])

In [7]:
print(X)

[[619 'delhi' 0 ... 1 1 101348.88]
 [608 'bangalore' 0 ... 0 1 112542.58]
 [502 'delhi' 0 ... 1 0 113931.57]
 ...
 [709 'delhi' 0 ... 0 1 42085.58]
 [772 'mumbai' 1 ... 1 0 92888.52]
 [792 'delhi' 0 ... 1 0 38190.78]]


In [8]:
df.head()

,apst,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,delhi,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,bangalore,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,delhi,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,delhi,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,bangalore,Female,43,2,125510.82,1,1,1,79084.10,0


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[
    ('encoder',OneHotEncoder(),[1])],remainder='passthrough')

X = np.array(ct.fit_transform(X))

In [10]:
X

array([[0.0, 1.0, 0.0, ..., 1, 1, 101348.88],
       [1.0, 0.0, 0.0, ..., 0, 1, 112542.58],
       [0.0, 1.0, 0.0, ..., 1, 0, 113931.57],
       ...,
       [0.0, 1.0, 0.0, ..., 0, 1, 42085.58],
       [0.0, 0.0, 1.0, ..., 1, 0, 92888.52],
       [0.0, 1.0, 0.0, ..., 1, 0, 38190.78]],
      shape=(10000, 12), dtype=object)

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=0)

In [12]:
# XGB
from xgboost import XGBClassifier
classifier = XGBClassifier(n_estimators = 25)
print(classifier)
classifier.fit(X_train,y_train)


y_pred = classifier.predict(X_test)
print()
print("y_pred :- ",y_pred)
print()
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test,y_pred)
print("Confusion Matrix :- \n",cm)
print()
print('accuracy score :- ',accuracy_score(y_test,y_pred))
print()
bias = classifier.score(X_train,y_train)
print('bias classifier :- ',bias)
print()
var = classifier.score(X_test,y_test)
print('variance classifier :- ',var)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=25,
              n_jobs=None, num_parallel_tree=None, ...)

y_pred :-  [0 0 0 ... 0 0 0]

Confusion Matrix :- 
 [[1508   87]
 [ 189  216]]

accuracy score :-  0.862

bias classifier :-  0.896625

variance classifier :-  0.862


In [13]:
# LightGBM
from lightgbm import LGBMClassifier
lgb = LGBMClassifier(boosting_type='dart', num_leaves = 31,
    max_depth = -1,
    learning_rate = 0.1,
    n_estimators = 100,
    subsample_for_bin = 200000)
lgb.fit(X_train,y_train)
y_pred = lgb.predict(X_test)
print()
print("y_pred :- ",y_pred)
print()
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test,y_pred)
print("Confusion Matrix :- \n",cm)
print()
print('accuracy score :- ',accuracy_score(y_test,y_pred))
print()
bias = lgb.score(X_train,y_train)
print('bias lgb :- ',bias)
print()
var = lgb.score(X_test,y_test)
print('variance lgb :- ',var)


[LightGBM] [Info] Number of positive: 1632, number of negative: 6368
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000239 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 858
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.204000 -> initscore=-1.361479
[LightGBM] [Info] Start training from score -1.361479

y_pred :-  [0 0 0 ... 0 0 0]

Confusion Matrix :- 
 [[1521   74]
 [ 196  209]]

accuracy score :-  0.865

bias lgb :-  0.89

variance lgb :-  0.865


## 13_OCT_2026

# AUC AND ROC